# BrainWeb · Tumor growth explorer

Explore **Original Fisher–KPP** or **Tumor + oxygen + necrosis** on the BrainWeb T1 ICBM normal 1 mm phantom. Start Jupyter from this folder with `uv run --locked jupyter lab tumor_growth_explorer.ipynb`, then **Run All**. Choose the model in **Model & oxygen**; Fisher–KPP remains the default.

The GUI starts at a 2 mm grid. Set **Δx = 1 mm** for native anatomy. Coordinates are RAS+ in mm: x right, y anterior, z superior. Images use neurological orientation (subject left on image left). The underlying phantom is normal; every tumor here is simulated.


## Models, units and assumptions

The original model is unchanged:

$$\partial_t c=\nabla\cdot(D\nabla c)+\rho c(1-c/K).$$

The coupled model adds necrotic material $z$ and quasi-steady oxygen $o$:

$$\partial_t c=\nabla\cdot(D\nabla c)+\rho g(o)c[1-(c+z)/K]_+-\delta h(o)c,$$
$$\partial_t z=\delta h(o)c-\lambda_z z,$$
$$0=\ell_o^2\nabla^2 o+p(1-o)-\beta(c/K)o,\qquad p=\frac{1}{1+\alpha(c+z)/K}.$$

Here $g(o)=\min(1,o/\theta_p)$, $h(o)=[1-o/\theta_d]_+$ and $[a]_+=\max(a,0)$. Require $0<\theta_d<\theta_p<1$.

- $c,z$: cells or necrotic cell equivalents/mm³; $D$: mm²/day; $\rho,\delta,\lambda_z$: day⁻¹; $K$: cells/mm³; $\ell_o$: mm. Oxygen is relative to the normal-tissue baseline, $o=1$. All other new coefficients are dimensionless. Defaults are illustrative and require calibration for biological interpretation.
- Necrotic tumor material is immobile, consumes no oxygen, and occupies growth capacity. It does not measure all damaged host tissue. $K$ regulates births, rather than imposing a hard packing constraint; incoming diffusion may exceed local capacity.
- The coupled model uses GM/WM/glial tissue with closed boundaries for cells and oxygen; CSF is excluded. The original model retains optional CSF transport and zero CSF proliferation. Class 8 uses GM coefficients. No treatment, deformation, explicit vascular dynamics, phenotype switching or DTI fiber anisotropy is modeled.
- Oxygen delivery decreases with total tumor material through a phenomenological perfusion closure. This is a modeling assumption, not measured vascular behavior. Set α to zero to compare unchanged perfusion.
- Seed profiles remain a uniform ellipsoid or a Gaussian truncated at 3σ, initially restricted to GM/WM/glial tissue.

### Numerical scheme

Both models retain conservative face fluxes with harmonic-mean diffusivity, no periodic wrapping, and the positivity bound $\Delta t\le0.9/\max_i\sum_j D_{ij}/\Delta x^2$. Original Fisher–KPP retains its exact local logistic flow.

The coupled solver applies diffusion, exact frozen-oxygen growth with remaining capacity, exact viable-to-necrotic transfer, and exponential clearance. Oxygen is solved at initialization and after each internal step using a compact, matrix-free, preconditioned conjugate-gradient solve. The relative residual tolerance defaults to $10^{-7}$; non-convergence raises an error. The method is first order in time. Density clipping is not used.

At millimeter resolution, oxygen represents voxel-averaged availability; the default 0.2 mm length and capillary-scale structures are not spatially resolved. Refine time steps and assess spatial resolution before interpreting quantitative results.

Viable and necrotic overlays use fixed $c/K$ and $z/K$ scales, hiding values below 0.02. Oxygen uses a fixed 0–1 scale, with only parenchyma colored. Volume thresholds are illustrative and are not calibrated MRI boundaries.


In [ ]:
%matplotlib widget
from IPython.display import display
from brainweb_kpp.data import verify_data
from brainweb_kpp.ui import TumorExplorer

manifest = verify_data()
app = TumorExplorer()
display(app.widget)

## Using the explorer

1. In **Origin & density**, place the seed, choose its profile, radii and peak density.
2. In **Diffusion & growth**, set diffusion, proliferation and carrying capacity.
3. In **Model & oxygen**, choose the original or coupled model. All seven biological coupling parameters have controls. Coupled mode can also run a Fisher–KPP reference with the same seed, grid and parenchymal transport settings.
4. In **Grid & time**, set resolution, maximum time step, duration and output cadence, then press **Run 3D simulation**.
5. Choose **Display → Side by side** to see coupled viable tumor, oxygen, necrosis and the optional Fisher–KPP reference in columns. Shared playback, frame and x/y/z slice controls update all panels. **Planes** selects one orientation or all three. The viable and reference panels use identical fixed color scales. Collapse **Simulation parameters & export** to make room.
6. For a single field, select **Display → Single field**, then select **Viable tumor**, **Oxygen availability**, **Necrotic material** or **Fisher–KPP reference** in **Result view**. Time and slice positions remain synchronized while switching fields. Changing model parameters clears the previous run.
7. **Export** writes the current display as MP4/GIF/poster and saves all available fields as NIfTI volumes and replay planes. Side-by-side movies keep all four fields synchronized at the displayed slice coordinates and plane selection. Single-field movies use the seed planes. Reference results have their own `reference_fisher_kpp/` subdirectory.

With 3D storage enabled, slice sliders explore every saved field. Without it, playback uses the seed planes. A native 1 mm float32 volume is about 28 MB. Coupled history stores three fields; the reference adds one more. At 61 frames, combined histories use about 6.8 GB before working arrays. An 8 GB history guard asks for coarser output. Uncheck 3D storage for lighter runs.

To see necrosis immediately in a short demonstration, use an 80,000 cells/mm³ Gaussian peak at K = 100,000. This changes the initial condition, not the default coupled equations. A batch example is in `examples/coupled.json`.


## Programmatic maps and batch runs

The same solver accepts custom arrays for D(x), ρ(x), and K(x), on the selected `brain` grid. Arrays must use the same canonical RAS axes. The notebook widgets use uniform tissue proliferation and uniform K. Custom spatial K can be exceeded locally by incoming diffusion; values are not clipped. Use the Python result arrays for such experiments: the standard media renderer assumes the scalar K in Parameters for its color scale.

The following cell is optional; it is left unexecuted to avoid starting a second simulation.

In [ ]:
# from dataclasses import replace
# from brainweb_kpp import Parameters, load_brainweb, simulate
# from brainweb_kpp.model import fields
# from brainweb_kpp.render import save_result, export_media
# brain = load_brainweb(spacing=1.0)
# p = replace(Parameters(), model="coupled", frame_days=1.5)
# D, rho, K = fields(brain, p)
# # Modify coefficient arrays here if needed.
# result = simulate(brain, p, diffusion=D, proliferation=rho, capacity=K, compare=True)
# save_result(result, "outputs/custom")
# export_media(result, "outputs/custom", fps=12)

## Sources and reproducibility

- [BrainWeb normal anatomical model and tissue classes](https://brainweb.bic.mni.mcgill.ca/anatomic_normal.html).
- [BrainWeb simulated database and citation information](https://brainweb.bic.mni.mcgill.ca/brainweb/).
- Collins et al. (1998), *Design and construction of a realistic digital brain phantom*, IEEE TMI 17(3), 463–468. [DOI](https://doi.org/10.1109/42.712135).
- Everett et al. (2020), *A tutorial review of mathematical techniques for quantifying tumor heterogeneity*. [DOI](https://doi.org/10.3934/mbe.2020207), as referenced in the supplied slides.
- Martínez-González et al. (2012), *Hypoxic Cell Waves Around Necrotic Cores in Glioblastoma*. [DOI](https://doi.org/10.1007/s11538-012-9786-1). Biological motivation; the reduced three-field closure here is a proposed simplification.

The bundled manifest records data checksums. `uv.lock` pins the environment. Each export records parameters, grid, actual numerical step, data provenance, library versions and metrics. See README.md for the independent animation command and numerical validation.